# Classic NLP Evaluation Metrics

This notebook covers the **reference-based** metrics used to compare a model's generated text against a ground-truth reference:

| Metric | Idea | Typical use |
|---|---|---|
| BLEU | n-gram precision overlap | translation |
| ROUGE-1/2/L | n-gram recall overlap | summarization |
| METEOR | overlap + synonym/stem matching + word order penalty | translation/summarization |
| BERTScore | embedding cosine similarity (contextual) | paraphrase-tolerant similarity |
| Exact Match / F1 | token-level overlap (SQuAD-style) | extractive QA |

We generate a **synthetic dataset** of (question, reference answer, candidate/model answer) triples that intentionally span:
- exact matches
- good paraphrases
- partially correct answers
- hallucinated/wrong answers

...so you can see how each metric reacts differently to the same failure modes.

In [1]:
# Install dependencies (uncomment if running fresh)
%pip install nltk rouge-score sacrebleu bert-score pandas matplotlib -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import nltk
import pandas as pd
import matplotlib.pyplot as plt

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

pd.set_option("display.max_colwidth", 120)

## 1. Synthetic dataset

Each row simulates one "ground truth vs model response" comparison, labeled with the failure mode it represents.

In [3]:
synthetic_data = [
    {
        "id": 1,
        "case": "exact_match",
        "question": "What is the capital of France?",
        "reference": "The capital of France is Paris.",
        "candidate": "The capital of France is Paris.",
    },
    {
        "id": 2,
        "case": "good_paraphrase",
        "question": "What is the capital of France?",
        "reference": "The capital of France is Paris.",
        "candidate": "Paris is France's capital city.",
    },
    {
        "id": 3,
        "case": "partially_correct",
        "question": "Summarize the causes of World War I.",
        "reference": "World War I was caused by militarism, alliances, imperialism, and nationalism, triggered by the assassination of Archduke Franz Ferdinand.",
        "candidate": "World War I started because of the assassination of Archduke Franz Ferdinand and rising nationalism in Europe.",
    },
    {
        "id": 4,
        "case": "hallucinated",
        "question": "What is the capital of France?",
        "reference": "The capital of France is Paris.",
        "candidate": "The capital of France is Lyon, a city known for its cuisine.",
    },
    {
        "id": 5,
        "case": "off_topic",
        "question": "What is the boiling point of water at sea level?",
        "reference": "Water boils at 100 degrees Celsius at sea level.",
        "candidate": "Water freezes at 0 degrees Celsius.",
    },
    {
        "id": 6,
        "case": "good_summary",
        "question": "Summarize: The stock market fell sharply today after the central bank raised interest rates, citing persistent inflation concerns.",
        "reference": "Stocks dropped after the central bank hiked interest rates due to inflation worries.",
        "candidate": "Markets declined following an interest rate hike from the central bank amid inflation concerns.",
    },
]

df = pd.DataFrame(synthetic_data)
df

,id,case,question,reference,candidate
0,1,exact_match,What is the capital of France?,The capital of France is Paris.,The capital of France is Paris.
1,2,good_paraphrase,What is the capital of France?,The capital of France is Paris.,Paris is France's capital city.
2,3,partially_correct,Summarize the causes of World War I.,"World War I was caused by militarism, alliances, imperialism, and nationalism, triggered by the assassination of Arc...",World War I started because of the assassination of Archduke Franz Ferdinand and rising nationalism in Europe.
3,4,hallucinated,What is the capital of France?,The capital of France is Paris.,"The capital of France is Lyon, a city known for its cuisine."
4,5,off_topic,What is the boiling point of water at sea level?,Water boils at 100 degrees Celsius at sea level.,Water freezes at 0 degrees Celsius.
5,6,good_summary,"Summarize: The stock market fell sharply today after the central bank raised interest rates, citing persistent infla...",Stocks dropped after the central bank hiked interest rates due to inflation worries.,Markets declined following an interest rate hike from the central bank amid inflation concerns.


## 2. BLEU

n-gram **precision**: what fraction of candidate n-grams appear in the reference, with a brevity penalty for short candidates. Sensitive to exact wording — paraphrases score poorly.

In [4]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

smoothie = SmoothingFunction().method1

def compute_bleu(reference: str, candidate: str) -> float:
    ref_tokens = [reference.lower().split()]
    cand_tokens = candidate.lower().split()
    return sentence_bleu(ref_tokens, cand_tokens, smoothing_function=smoothie)

df["bleu"] = df.apply(lambda r: compute_bleu(r["reference"], r["candidate"]), axis=1)
df[["case", "reference", "candidate", "bleu"]]

,case,reference,candidate,bleu
0,exact_match,The capital of France is Paris.,The capital of France is Paris.,1.000000
1,good_paraphrase,The capital of France is Paris.,Paris is France's capital city.,0.052312
2,partially_correct,"World War I was caused by militarism, alliances, imperialism, and nationalism, triggered by the assassination of Arc...",World War I started because of the assassination of Archduke Franz Ferdinand and rising nationalism in Europe.,0.262168
3,hallucinated,The capital of France is Paris.,"The capital of France is Lyon, a city known for its cuisine.",0.317023
4,off_topic,Water boils at 100 degrees Celsius at sea level.,Water freezes at 0 degrees Celsius.,0.032588
5,good_summary,Stocks dropped after the central bank hiked interest rates due to inflation worries.,Markets declined following an interest rate hike from the central bank amid inflation concerns.,0.080323


## 3. ROUGE (1 / 2 / L)

n-gram **recall**-oriented: how much of the reference's content is captured by the candidate. ROUGE-L uses longest common subsequence, tolerant of word order.

In [5]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

def compute_rouge(reference: str, candidate: str) -> dict:
    scores = scorer.score(reference, candidate)
    return {
        "rouge1_f": scores["rouge1"].fmeasure,
        "rouge2_f": scores["rouge2"].fmeasure,
        "rougeL_f": scores["rougeL"].fmeasure,
    }

rouge_results = df.apply(lambda r: compute_rouge(r["reference"], r["candidate"]), axis=1)
df = pd.concat([df, pd.DataFrame(list(rouge_results))], axis=1)
df[["case", "rouge1_f", "rouge2_f", "rougeL_f"]]

,case,rouge1_f,rouge2_f,rougeL_f
0,exact_match,1.000000,1.000000,1.000000
1,good_paraphrase,0.666667,0.000000,0.166667
2,partially_correct,0.611111,0.411765,0.500000
3,hallucinated,0.555556,0.500000,0.555556
4,off_topic,0.533333,0.153846,0.533333
5,good_summary,0.518519,0.240000,0.296296


## 4. METEOR

Improves on BLEU by matching synonyms/stems (via WordNet) and adding a word-order penalty. Harmonic mean weighted toward recall — correlates better with human judgment than BLEU on single sentences.

In [6]:
from nltk.translate.meteor_score import meteor_score
from nltk.tokenize import word_tokenize

def compute_meteor(reference: str, candidate: str) -> float:
    ref_tokens = word_tokenize(reference.lower())
    cand_tokens = word_tokenize(candidate.lower())
    return meteor_score([ref_tokens], cand_tokens)

df["meteor"] = df.apply(lambda r: compute_meteor(r["reference"], r["candidate"]), axis=1)
df[["case", "bleu", "rougeL_f", "meteor"]]

,case,bleu,rougeL_f,meteor
0,exact_match,1.000000,1.000000,0.998542
1,good_paraphrase,0.052312,0.166667,0.357143
2,partially_correct,0.262168,0.500000,0.494272
3,hallucinated,0.317023,0.555556,0.764791
4,off_topic,0.032588,0.533333,0.383505
5,good_summary,0.080323,0.296296,0.498116


## 5. BERTScore

Instead of token overlap, embeds both texts with a contextual model (e.g. RoBERTa) and computes cosine similarity between matched tokens. Catches paraphrases that BLEU/ROUGE penalize.

> First run downloads a ~400MB model — expect a delay on first execution.

In [ ]:
from bert_score import score as bert_score_fn

P, R, F1 = bert_score_fn(
    df["candidate"].tolist(),
    df["reference"].tolist(),
    lang="en",
    verbose=False,
)

df["bertscore_precision"] = P.tolist()
df["bertscore_recall"] = R.tolist()
df["bertscore_f1"] = F1.tolist()
df[["case", "bleu", "meteor", "bertscore_f1"]]

Notice case `good_paraphrase`: BLEU/ROUGE score it low (little lexical overlap) while BERTScore scores it high (semantically equivalent) — this is exactly why BERTScore is preferred when wording is expected to vary.

## 6. Exact Match / F1 (SQuAD-style)

Standard extractive-QA metrics: normalize both strings (lowercase, strip punctuation/articles), then compare exactly (EM) or as a token-level F1 (bag-of-words overlap).

In [ ]:
import re
import string
from collections import Counter

def normalize_answer(s: str) -> str:
    s = s.lower()
    s = "".join(ch for ch in s if ch not in string.punctuation)
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())

def exact_match(reference: str, candidate: str) -> int:
    return int(normalize_answer(reference) == normalize_answer(candidate))

def token_f1(reference: str, candidate: str) -> float:
    ref_tokens = normalize_answer(reference).split()
    cand_tokens = normalize_answer(candidate).split()
    common = Counter(ref_tokens) & Counter(cand_tokens)
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0
    precision = num_same / len(cand_tokens)
    recall = num_same / len(ref_tokens)
    return 2 * precision * recall / (precision + recall)

df["exact_match"] = df.apply(lambda r: exact_match(r["reference"], r["candidate"]), axis=1)
df["token_f1"] = df.apply(lambda r: token_f1(r["reference"], r["candidate"]), axis=1)
df[["case", "exact_match", "token_f1"]]

## 7. All metrics side by side

In [ ]:
summary_cols = [
    "case", "bleu", "rouge1_f", "rouge2_f", "rougeL_f", "meteor",
    "bertscore_f1", "exact_match", "token_f1",
]
summary = df[summary_cols].set_index("case").round(3)
summary

In [ ]:
ax = summary.drop(columns=["exact_match"]).plot(kind="bar", figsize=(12, 5))
ax.set_title("Classic NLP metrics across synthetic cases")
ax.set_ylabel("score")
ax.set_ylim(0, 1)
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## Takeaways

- **`good_paraphrase`**: low BLEU/ROUGE, high BERTScore/METEOR — lexical-overlap metrics punish valid paraphrases.
- **`hallucinated`**: shares surface words ("capital of France is") so BLEU/ROUGE aren't near-zero even though the fact is wrong — string-overlap metrics cannot detect factual errors, only lexical similarity.
- **`off_topic`**: all metrics correctly collapse toward 0.
- **`exact_match`**: everything saturates at 1.0, as expected.

This is exactly why production agentic/RAG systems layer on **semantic and faithfulness-aware** evaluation (BERTScore, and further, LLM-as-judge frameworks like RAGAS) rather than relying on BLEU/ROUGE alone — see `02_ragas_evaluation.ipynb`.